## Imports

In [3]:
# Imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")

# Imports for production processing pipeline
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

# Baseline dummy classiier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (confusion_matrix, precision_score, accuracy_score, recall_score, f1_score, classification_report)

# set the display to max column to see all columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.4f}".format)

# consistent plotting style
plt.style.use("default")

In [4]:
# Coniguration
RANDOM_STATE = 42
TEST_SIZE = 0.20  

In [5]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
DATA_PATH = r"C:\Projects\RTFD\data\00_raw\PS_20174392719_1491204439457_log.csv"

df = pd.read_csv(DATA_PATH)

# verify
print(f"Dataset Shape : {df.shape}")

display(df.head())

df.info()

Dataset Shape : (6362620, 11)


,step,type,amount,nameOrig,oldbalanceOrg,newbalanceOrig,nameDest,oldbalanceDest,newbalanceDest,isFraud,isFlaggedFraud
0,1,PAYMENT,9839.6400,C1231006815,170136.0000,160296.3600,M1979787155,0.0000,0.0000,0,0
1,1,PAYMENT,1864.2800,C1666544295,21249.0000,19384.7200,M2044282225,0.0000,0.0000,0,0
2,1,TRANSFER,181.0000,C1305486145,181.0000,0.0000,C553264065,0.0000,0.0000,1,0
3,1,CASH_OUT,181.0000,C840083671,181.0000,0.0000,C38997010,21182.0000,0.0000,1,0
4,1,PAYMENT,11668.1400,C2048537720,41554.0000,29885.8600,M1230701703,0.0000,0.0000,0,0


<class 'pandas.DataFrame'>
RangeIndex: 6362620 entries, 0 to 6362619
Data columns (total 11 columns):
 #   Column          Dtype  
---  ------          -----  
 0   step            int64  
 1   type            str    
 2   amount          float64
 3   nameOrig        str    
 4   oldbalanceOrg   float64
 5   newbalanceOrig  float64
 6   nameDest        str    
 7   oldbalanceDest  float64
 8   newbalanceDest  float64
 9   isFraud         int64  
 10  isFlaggedFraud  int64  
dtypes: float64(5), int64(3), str(3)
memory usage: 534.0 MB


In [6]:
# Dataset Verification
print("-" * 60)
print("Dataset Verification")
print("-" * 60)

print(f"shape       : {df.shape}")
print(f"Memory Usage: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")
print("\nColumns: ")
print(df.columns.to_list())

print("\nMissing values:")
print(df.isnull().sum())

print("\nData Types: ")
print(df.dtypes)

------------------------------------------------------------
Dataset Verification
------------------------------------------------------------
shape       : (6362620, 11)
Memory Usage: 1598.19 MB

Columns: 
['step', 'type', 'amount', 'nameOrig', 'oldbalanceOrg', 'newbalanceOrig', 'nameDest', 'oldbalanceDest', 'newbalanceDest', 'isFraud', 'isFlaggedFraud']

Missing values:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Data Types: 
step                int64
type                  str
amount            float64
nameOrig              str
oldbalanceOrg     float64
newbalanceOrig    float64
nameDest              str
oldbalanceDest    float64
newbalanceDest    float64
isFraud             int64
isFlaggedFraud      int64
dtype: object


## Dataset Preparation

### Why are we doing this?
- Imagine you're working in the bank as an engineer
- every second, a new transaction arrives:
    | Feature          | Value    |
    | ---------------- | -------- |
    | Step             | 357      |
    | Type             | TRANSFER |
    | Amount           | ₹450,000 |
    | Sender Balance   | ₹600,000 |
    | Receiver Balance | ₹20,000  |
The fraud investigation team asks:
```
Can your model tell us whether this transaction is fraudulent?
```
### Input Feature(X):
These are the pieces of information the model used to learn the pattern.\
For PaySim, we selected:
| Feature          | Why?                                                            |
| ---------------- | --------------------------------------------------------------- |
| `step`           | Fraud patterns may vary over time.                              |
| `type`           | Your EDA showed fraud only occurs in `TRANSFER` and `CASH_OUT`. |
| `amount`         | Fraudulent transactions are generally much larger.              |
| `oldbalanceOrg`  | Sender's balance before the transaction.                        |
| `newbalanceOrig` | Sender's balance after the transaction.                         |
| `oldbalanceDest` | Receiver's balance before the transaction.                      |
| `newbalanceDest` | Receiver's balance after the transaction.                       |

This Becomes: ```X```

### Target Value(Y):
The target is the value we want the model to predict
for this project ``` y = isFraud``` is the target value

### Why ```isFraud``` is Not used?

While doing EDA we discover:
```
Precision = 100%
Recall = 0.19%
```
This means it comes from an existing rule engine.

If we include it now, the model would be learning from another fraud detector instead of learning directly from the transaction behaviour. So Exclude it..

Later we will build hybrid system:
```text
Business Rules
        +
Machine Learning
```



In [7]:
# Select Features and Target variable

# X - input features
feature_columns = [
    "step",
    "type",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]

x = df[feature_columns]

# Y - target value

y = df["isFraud"]

In [8]:
# Verify Feature Matrix and Target Variable

print("=" * 60)
print("Feature Matrix (X)")
print("=" * 60)

print(f"Shape: {x.shape}")

display(x.head())

print("\n")

print("=" * 60)
print("Target Variable (y)")
print("=" * 60)

print(f"Shape: {y.shape}")

display(y.value_counts())

Feature Matrix (X)
Shape: (6362620, 7)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
0,1,PAYMENT,9839.6400,170136.0000,160296.3600,0.0000,0.0000
1,1,PAYMENT,1864.2800,21249.0000,19384.7200,0.0000,0.0000
2,1,TRANSFER,181.0000,181.0000,0.0000,0.0000,0.0000
3,1,CASH_OUT,181.0000,181.0000,0.0000,21182.0000,0.0000
4,1,PAYMENT,11668.1400,41554.0000,29885.8600,0.0000,0.0000




Target Variable (y)
Shape: (6362620,)


isFraud
0    6354407
1       8213
Name: count, dtype: int64

## Train Test Split
Basically Train, Test split is to chek ```the model is learning the pattern or memorizing the data.```

Train/Test_split should be **80/20%**
If the data is memorizing means it is ```overfitting```

### Overfitting
```Overfitting``` is when the ```model memorizes the seen data instead of learning the patterns.```

### Genralization
The true goal of the ```machine learning``` is not to perform well on the data data.
The goal is:
```text
Past Transactions
        ↓
Learn Patterns
        ↓
Future Transactions
```
### Training v/s Testing:
To measure the ```generalization```, we divide the dataset into 2 parts:
```text
Entire Dataset
        │
        ▼
 ┌──────────────┐
 │ Training Set │
 └──────────────┘
        │
 Model Learns
        │
        ▼
 ┌──────────────┐
 │ Testing Set  │
 └──────────────┘
 ```

In [9]:
# Verify Feature Matrix and Target Variable

print("=" * 60)
print("Feature Matrix (X)")
print("=" * 60)

print(f"Shape: {x.shape}")

display(x.head())

print("\n")

print("=" * 60)
print("Target Variable (y)")
print("=" * 60)

print(f"Shape: {y.shape}")

display(y.value_counts())

Feature Matrix (X)
Shape: (6362620, 7)


,step,type,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest
0,1,PAYMENT,9839.6400,170136.0000,160296.3600,0.0000,0.0000
1,1,PAYMENT,1864.2800,21249.0000,19384.7200,0.0000,0.0000
2,1,TRANSFER,181.0000,181.0000,0.0000,0.0000,0.0000
3,1,CASH_OUT,181.0000,181.0000,0.0000,21182.0000,0.0000
4,1,PAYMENT,11668.1400,41554.0000,29885.8600,0.0000,0.0000




Target Variable (y)
Shape: (6362620,)


isFraud
0    6354407
1       8213
Name: count, dtype: int64

In [10]:
# Train, Test Split
x_train, x_test, y_train, y_test = train_test_split(
                                                    x,
                                                    y,
                                                    test_size=0.20, # 80% training and 20% testing
                                                    random_state=42, # for best result, its simply fixed seed for reproducibility.
                                                    stratify=y # to equally distributes the y target value to both train and test split data 
                                                    )

In [11]:
# Validation
print("-"*60)
print("train_test_split validation")
print("-"*60)

print(f"\nTraining shape : {x_train.shape}")
print(f"\nTesting shape : {x_test.shape}")

print("\nFraud Distribution (Training)")
print(y_train.value_counts(normalize=True))

print("\n")

print("=" * 60)
print("Testing Set")
print("=" * 60)

print(f"X_test Shape : {x_test.shape}")
print(f"y_test Shape : {y_test.shape}")

print("\nFraud Distribution (Testing)")
print(y_test.value_counts(normalize=True))

------------------------------------------------------------
train_test_split validation
------------------------------------------------------------

Training shape : (5090096, 7)

Testing shape : (1272524, 7)

Fraud Distribution (Training)
isFraud
0   0.9987
1   0.0013
Name: proportion, dtype: float64


Testing Set
X_test Shape : (1272524, 7)
y_test Shape : (1272524,)

Fraud Distribution (Testing)
isFraud
0   0.9987
1   0.0013
Name: proportion, dtype: float64


## Production Preprocessing Pipeline

---

## Learning Goal

In this module, we will prepare our features for machine learning by building a reusable preprocessing pipeline.

By the end of this module, you will understand:

- Why raw data cannot always be used directly by machine learning models.
- The difference between numerical and categorical features.
- Why categorical features require encoding.
- What One-Hot Encoding is and why it is preferred for nominal features.
- What `ColumnTransformer` and `Pipeline` are.
- Why preprocessing should be performed **inside** a machine learning pipeline.
- How pipelines help prevent **data leakage** and improve reproducibility.

---

## Why Do We Need Preprocessing?

Machine learning models work with **numerical data**, but our dataset contains both numerical and categorical features.

For example, the `type` column contains values such as:

- PAYMENT
- TRANSFER
- CASH_OUT
- CASH_IN
- DEBIT

A machine learning algorithm cannot directly interpret these text values. Therefore, they must be transformed into a numerical representation before training.

Similarly, production datasets often contain missing values. Even though the PaySim dataset has no missing values, it is considered good practice to include missing-value handling in the preprocessing pipeline so that the workflow is robust to future data.

---

## Why Not Encode the Data Manually?

A common beginner approach is to manually encode the dataset before training the model.

For example:

```python
encoder.fit(df["type"])
```

Although this may appear to work, it introduces poor workflow practices and can lead to **data leakage** when preprocessing steps learn information from the entire dataset before evaluation.

Instead, we first split the dataset into training and testing sets, then allow the preprocessing pipeline to learn **only from the training data**.

This ensures that the test dataset remains completely unseen during training, giving us a realistic estimate of model performance.

---

## Components of Our Preprocessing Pipeline

### Numerical Features

The following features are already numeric:

- `step`
- `amount`
- `oldbalanceOrg`
- `newbalanceOrig`
- `oldbalanceDest`
- `newbalanceDest`

For these features, we will apply:

- **SimpleImputer (Median Strategy)**

Although the PaySim dataset has no missing values, including an imputer makes the pipeline production-ready.

---

### Categorical Features

The categorical feature is:

- `type`

This feature will be processed using:

- **SimpleImputer (Most Frequent Strategy)**
- **OneHotEncoder**

One-Hot Encoding converts each transaction type into a separate binary feature, avoiding the incorrect assumption that categories have a natural numerical order.

---

## ColumnTransformer

Our dataset contains multiple feature types.

Instead of preprocessing every column manually, `ColumnTransformer` allows us to apply different preprocessing steps to different groups of columns.

This keeps the preprocessing workflow organized, reusable, and scalable.

---

## Pipeline

Finally, the preprocessing steps will later be combined with the machine learning model using a Scikit-learn `Pipeline`.

The workflow will look like this:

```text
Raw Transaction
        │
        ▼
Preprocessing
    ├── Missing Value Handling
    ├── One-Hot Encoding
        │
        ▼
Machine Learning Model
        │
        ▼
Fraud Probability
        │
        ▼
Prediction
```

Using a pipeline ensures that every new transaction is processed in exactly the same way as the training data.

This is the standard approach used in production machine learning systems because it improves reproducibility, prevents data leakage, and simplifies deployment.

In [12]:
# Defining Numerical and categoical 

numerical_features = [
    "step",
    "amount",
    "oldbalanceOrg",
    "newbalanceOrig",
    "oldbalanceDest",
    "newbalanceDest"
]


# Categorical features

categorical_features = [
    "type"
]

In [13]:
# Numerical Pipeline
numerical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)
# Categorical Preprocessing
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [14]:
# Build the Preprocessor
# Column Transformer

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

In [16]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numerical_transformer, numerical_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

## Baseline Dummy Model# Module 4 — Dummy Baseline Model

## Learning Goal

Before training a real machine learning model, we first build a **Dummy Baseline Model**.

The purpose is **not** to detect fraud, but to create a **minimum benchmark** that every future model must outperform.

---

## Why Do We Need a Dummy Model?

The PaySim dataset is **highly imbalanced**:

* **99.87%** Legitimate transactions
* **0.13%** Fraudulent transactions

Imagine a model that simply predicts:

> **"Every transaction is legitimate."**

This model would still achieve approximately **99.87% accuracy**, even though it detects **zero frauds**.

This shows that **accuracy alone is not a reliable metric** for fraud detection.

---

## Why Are We Building It?

The Dummy Model helps us answer an important question:

> **"Is our machine learning model actually learning useful fraud patterns, or is it only benefiting from the class imbalance?"**

If a Logistic Regression or Random Forest cannot perform better than this simple baseline, then the model is not adding any value.

---

## Expected Outcome

The Dummy Classifier will:

* Predict every transaction as **Legitimate**
* Achieve **very high Accuracy (~99.87%)**
* Have **0 Precision**
* Have **0 Recall**
* Have **0 F1-score**

This demonstrates why fraud detection focuses on **Precision, Recall, F1-score, and PR-AUC** instead of Accuracy.

---

## Production Perspective

In real-world machine learning projects, every new model is compared against a baseline before deployment.

The typical evaluation flow is:

```text
Dummy Baseline
        ↓
Logistic Regression
        ↓
Random Forest
        ↓
LightGBM
        ↓
Production Model
```

A new model should only be considered useful if it clearly outperforms the baseline.


In [17]:
dummy_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classiier", DummyClassifier(strategy="most_frequent"))
    ]
)


In [20]:
# Train Dummy Model

dummy_pipeline.fit(x_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('classiier', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](7,)","['step','type','amount',...,'newbalanceOrig','oldbalanceDest', 'newbalanceDest']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,7
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrou

In [22]:
# Dummy Predictions
y_pred_dummy = dummy_pipeline.predict(x_test)

In [23]:
# ============================================================
# Evaluate Dummy Model
# ============================================================

print("=" * 60)
print("Dummy Classifier Results")
print("=" * 60)

print(f"Accuracy : {accuracy_score(y_test, y_pred_dummy):.4f}")
print(f"Precision: {precision_score(y_test, y_pred_dummy, zero_division=0):.4f}")
print(f"Recall   : {recall_score(y_test, y_pred_dummy):.4f}")
print(f"F1 Score : {f1_score(y_test, y_pred_dummy):.4f}")

print("\nClassification Report")
print(classification_report(y_test, y_pred_dummy, zero_division=0))

Dummy Classifier Results
Accuracy : 0.9987
Precision: 0.0000
Recall   : 0.0000
F1 Score : 0.0000

Classification Report
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1270881
           1       0.00      0.00      0.00      1643

    accuracy                           1.00   1272524
   macro avg       0.50      0.50      0.50   1272524
weighted avg       1.00      1.00      1.00   1272524

